### Vanilla GAN For Celeba dataset

In [2]:
import torch 
import torch.nn as nn 
import torchvision
import os
import torch.optim as optim 
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image

In [3]:
# load image => transform image => dataset 

class ImageProcessor:
    def __init__(self,root_dir_path,transformations=None):

        self.root_dir_path=root_dir_path
        self.transformations=transformations

        self.all_img_paths=[os.path.join(root_dir_path,img) for img in os.listdir(root_dir_path)]

    def __len__(self):
        return len(self.all_img_paths)

    def __getitem__(self,idx):
        img_path=self.all_img_paths[idx]
        img=Image.open(img_path).convert("RGB")

        if self.transformations:
            img=self.transformations(img)

        return img


In [4]:
root_dir="./img_align_celeba"

In [5]:
transformations=transforms.Compose(
    [
        transforms.CenterCrop(178), # 178x218 => 178x178
        transforms.Resize(64) ,
        transforms.ToTensor(),
        transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) # [-1,1]
    ]
)

In [6]:
dataset=ImageProcessor(root_dir,transformations)
print(f"Loaded {len(dataset)} images")

Loaded 202597 images


In [7]:
# DataLoader

dataloader=DataLoader(dataset,batch_size=128,shuffle=True)

In [8]:
import numpy as np

### Generator Network 

In [9]:
class Generator(nn.Module):
    def __init__(self,z_dim=100,img_channels=3):
        super(Generator,self).__init__()

        # Fully connected layer

        self.model=nn.Sequential(
            nn.Linear(z_dim,256),
            nn.ReLU(),

            nn.Linear(256,512),
            nn.ReLU(),

            nn.Linear(512,1024),
            nn.ReLU(),

            nn.Linear(1024,64*64*3),
            nn.Tanh()
        )

    def forward(self,z):
        img=self.model(z)
        img=img.view(img.size(0),3,64,64)
        return img


In [10]:
generator=Generator()

### Discriminator Network

In [11]:
class Discriminator(nn.Module):
    def __init__(self,img_channels=3):
        super(Discriminator,self).__init__()

        self.model=nn.Sequential(
            nn.Flatten(),

            nn.Linear(img_channels*64*64,1024),
            nn.LeakyReLU(0.2,inplace=True),

            nn.Linear(1024,512),
            nn.LeakyReLU(0.2,inplace=True),

            nn.Linear(512,256),
            nn.LeakyReLU(0.2,inplace=True),

            nn.Linear(256,1),
            nn.Sigmoid()
        )

    def forward(self,img):
        return self.model(img)

In [12]:
discriminator=Discriminator()

In [13]:
GAN_loss=nn.BCELoss()

In [14]:
generator_optimizer=optim.Adam(generator.parameters(),lr=0.0002,betas=(0.5,0.999)) 
discriminator_optimizer=optim.Adam(discriminator.parameters(),lr=0.0002,betas=(0.5,0.999)) 

In [15]:
# Device

if torch.backends.mps.is_available():
    device=torch.device("mps")
elif torch.cuda.is_available():
    device=torch.device("cuda")
else:
    device=torch.device("cpu")

In [16]:
print(device)

cuda


In [17]:
generator=generator.to(device)

In [18]:
discriminator=discriminator.to(device)

### Training the GAN

In [24]:
def train(generator,discriminator,dataloader,epochs=5):

    for epoch in range(epochs):

        for i , imgs in enumerate(dataloader):
            real_imgs=imgs.to(device)
            batch_size=real_imgs.size(0)

            # Create real img labels and fake img labels

            real_labels=torch.ones(batch_size,1).to(device)
            fake_labels=torch.zeros(batch_size,1).to(device)

            # Train the discriminator

            discriminator_optimizer.zero_grad()

            fake_imgs=generator(torch.randn(batch_size,100).to(device))

            real_loss=GAN_loss(discriminator(real_imgs),real_labels)

            fake_loss=GAN_loss(discriminator(fake_imgs.detach()),fake_labels)

            d_loss=(real_loss+fake_loss)/2

            d_loss.backward()

            discriminator_optimizer.step()

            # Train the generator

            generator_optimizer.zero_grad()

            g_loss=GAN_loss(discriminator(fake_imgs),real_labels)

            g_loss.backward()

            generator_optimizer.step()


            if i%50==0:
                print(f"{epoch+1}/{epochs} => batch {i+1} => G-Loss is {g_loss} & D Loss is {d_loss}")

        save_generated_images(generator,epoch,device)

In [25]:
import matplotlib.pyplot as plt 

In [28]:
def save_generated_images(generator,epoch,device,num_imgs=8):
    z=torch.randn(num_imgs,100).to(device)
    generated_imgs=generator(z.detach()).cpu()

    grid=torchvision.utils.make_grid(
        generated_imgs,nrow=4,normalize=True
    )

    plt.imshow(np.transpose(grid,(1,2,0)))
    plt.title(f"Epoch {epoch+1}")

    plt.axis("off")
    plt.show()

In [ ]:
train(generator,discriminator,dataloader,epochs=2)

1/2 => batch 1 => G-Loss is 3.675326347351074 & D Loss is 0.04456041008234024
1/2 => batch 51 => G-Loss is 3.6507129669189453 & D Loss is 0.0933375284075737
